<h3>DOCX·HTML·CSV 실습 준비</h3>
<p><strong>개념</strong> DOCX는 문단, HTML은 태그, CSV는 행과 열을 보존합니다. 형식별 객체를 먼저 만든 뒤 각 구조에서 필요한 값을 꺼냅니다.</p>

In [1]:
# %pip은 현재 커널에 HTML과 DOCX 로더를 설치합니다.
# CSV는 Python 표준 라이브러리이므로 별도 패키지를 설치하지 않습니다.
%pip install -q "beautifulsoup4==4.15.0" "python-docx==1.2.0"


Note: you may need to restart the kernel to use updated packages.


<h3>실행 위치와 제공 파일 확인</h3>
<p>이 노트북은 Ch10의 <code>fixtures</code> 폴더에 있는 작은 DOCX·HTML·CSV 파일을 사용합니다. 세 파일을 찾지 못하면 내용을 읽기 전에 중단합니다. API 키는 필요하지 않습니다.</p>

In [2]:
# 형식별 로더와 경로 도구를 가져옵니다.
# 반복문으로 상위 폴더를 확인해 Ch10 실습 폴더를 찾습니다.
from pathlib import Path
from bs4 import BeautifulSoup
from docx import Document
import csv

lab = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    possible = candidate / "labs/part3/ch10-document-processing"
    if possible.is_dir():
        lab = possible
        break
if lab is None:
    raise FileNotFoundError("저장소 안에서 노트북을 실행하세요.")
fixtures = lab / "fixtures"
paths = [fixtures / "sample.docx", fixtures / "sample.html", fixtures / "sample.csv"]
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.name, path.stat().st_size, "bytes")


sample.docx 36857 bytes
sample.html 102 bytes
sample.csv 48 bytes


<h3>DOCX 객체와 문단 관찰</h3>
<p><code>Document</code> 객체의 <code>paragraphs</code> 속성은 문단 객체 목록입니다. 각 문단의 <code>text</code> 속성에서 실제 문자열을 읽습니다.</p>

In [3]:
# DOCX 파일을 열어 Document 객체와 전체 문단 수를 먼저 확인합니다.
# 빈 문단은 제외하고 텍스트가 있는 문단만 paragraphs에 추가합니다.
docx = Document(fixtures / "sample.docx")
paragraphs = []
for paragraph in docx.paragraphs:
    if paragraph.text.strip():
        paragraphs.append(paragraph.text)
print("객체:", type(docx).__name__)
print("전체 문단:", len(docx.paragraphs))
print("본문 문단:", paragraphs)


객체: Document
전체 문단: 2
본문 문단: ['배송 안내', '제주 배송에는 이틀이 더 필요합니다.']


<h3>HTML 태그와 본문 관찰</h3>
<p><code>BeautifulSoup</code> 객체는 HTML 태그 구조를 탐색합니다. <code>style</code>처럼 화면 표시를 설정하는 태그를 제거한 뒤 제목과 문단 텍스트만 확인합니다.</p>

In [4]:
# read_text()로 HTML 문자열을 읽고 html.parser로 태그 객체를 만듭니다.
# find_all()은 이름이 h1 또는 p인 태그를 문서 순서대로 반환합니다.
html_source = (fixtures / "sample.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html_source, "html.parser")
for tag in soup(["script", "style"]):
    tag.decompose()
content = []
for tag in soup.find_all(["h1", "p"]):
    content.append(tag.get_text(" ", strip=True))
print("객체:", type(soup).__name__)
print("선택한 본문:", content)


객체: BeautifulSoup
선택한 본문: ['반품 안내', '수령 후 7일 이내 신청합니다.']


<h3>CSV 행과 열 관찰</h3>
<p><code>DictReader</code>는 첫 줄을 열 이름으로 사용하고 이후 각 행을 딕셔너리로 반환합니다. 따라서 <code>rows[0]["상품"]</code>처럼 첫 행의 특정 열을 읽을 수 있습니다.</p>

In [5]:
# utf-8-sig는 파일 앞의 BOM이 열 이름에 섞이지 않도록 처리합니다.
# list()는 DictReader가 차례로 반환하는 행을 저장해 다시 확인할 수 있게 합니다.
with (fixtures / "sample.csv").open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    rows = list(reader)
print("열 이름:", columns)
print("행 개수:", len(rows))
print("첫 행:", rows[0])


열 이름: ['상품', '보관']
행 개수: 2
첫 행: {'상품': '쿠키', '보관': '실온'}


<h3>선택값 하나를 바꾸어 비교</h3>
<p>같은 첫 번째 CSV 행에서 선택하는 열만 <code>상품</code>에서 <code>보관</code>으로 바꿉니다. 행 객체는 같지만 열 이름에 따라 반환되는 값이 달라집니다.</p>

In [6]:
# row는 첫 번째 CSV 행을 나타내는 딕셔너리입니다.
# 대괄호 안의 열 이름만 바꾸어 두 값을 비교합니다.
row = rows[0]
product = row["상품"]
storage = row["보관"]
print("상품 열:", product)
print("보관 열:", storage)


상품 열: 쿠키
보관 열: 실온


<h3>실패 확인</h3>
<p><strong>Before 원인</strong> CSV 행에서 값만 꺼내면 <code>상품</code>과 <code>보관</code>이라는 열 이름이 사라집니다. 그러면 각 값이 무엇을 뜻하는지 이름으로 확인할 수 없습니다.</p>

In [7]:
# values_only에는 값의 순서만 남고 열 이름은 남지 않습니다.
values_only = [list(row.values()) for row in rows]
print("열 이름을 버린 결과:", values_only[0])

try:
    assert "상품" in values_only[0] and "보관" in values_only[0]
except AssertionError:
    print("caught: 값만으로는 상품과 보관 방법을 구분할 수 없습니다.")

열 이름을 버린 결과: ['쿠키', '실온']
caught: 값만으로는 상품과 보관 방법을 구분할 수 없습니다.


<p><strong>After 수정</strong> 각 행을 딕셔너리로 유지하면 열 이름과 값의 관계가 남습니다. 첫 행에서 상품과 보관 방법을 이름으로 읽어 함께 출력합니다.</p>

In [8]:
# DictReader가 만든 딕셔너리는 열 이름과 값을 한 쌍으로 보존합니다.
first_row = rows[0]
product = first_row["상품"]
storage = first_row["보관"]

assert first_row == {"상품": "쿠키", "보관": "실온"}
print(f"{product}의 보관 방법: {storage}")

쿠키의 보관 방법: 실온
